# Hidden elements are difficult to detect

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import median_filter
import timm
from pathlib import Path

# Configuration
class CFG:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    image_path = "/kaggle/input/csiro-biomass/train/ID1012260530.jpg"
    n_clusters = 8
    patch_size = 14
    input_size = 518
    upsample_factor = 4
    
    # Feature weights
    dino_weight = 0.4
    color_weight = 0.3
    veg_index_weight = 0.2
    texture_weight = 0.1
    
def load_dino_backbone():
    model = timm.create_model(
        "vit_base_patch14_dinov2",
        pretrained=True,
        num_classes=0,
    )
    model.eval()
    model.to(CFG.device)
    print(f"Loaded DINOv2 Base | device={CFG.device}")
    return model

def extract_patch_features(model, image):
    img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (CFG.input_size, CFG.input_size))
    
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_norm = (img / 255.0 - mean) / std
    
    img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).float().unsqueeze(0)
    img_tensor = img_tensor.to(CFG.device)
    
    with torch.no_grad():
        features = model.forward_features(img_tensor)
        patch_tokens = features[:, 1:, :]
        
    n_patches = patch_tokens.shape[1]
    grid_h = grid_w = int(np.sqrt(n_patches))
    
    return patch_tokens.squeeze(0).cpu().numpy(), (grid_h, grid_w), img

def compute_vegetation_indices(image):
    """Compute vegetation indices with CORRECT normalization."""
    img = image.astype(np.float32) / 255.0
    R, G, B = img[:, :, 0], img[:, :, 1], img[:, :, 2]
    eps = 1e-6
    
    # ExG range: approximately [-2, +2] for RGB in [0,1]
    ExG = 2*G - R - B
    
    # ExGR 
    ExR = 1.4*R - G
    ExGR = ExG - ExR
    
    # NGRDI: already in [-1, 1]
    NGRDI = (G - R) / (G + R + eps)
    
    # GLI: already in [-1, 1]
    GLI = (2*G - R - B) / (2*G + R + B + eps)
    
    # VARI
    VARI = (G - R) / (G + R - B + eps)
    
    # TGI range: approximately [-0.6, +0.6]
    TGI = G - 0.39*R - 0.61*B
    
    return {
        'ExG': ExG,
        'ExGR': ExGR,
        'NGRDI': NGRDI,
        'GLI': GLI,
        'VARI': VARI,
        'TGI': TGI
    }

def compute_texture_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)
    kernel_size = 5
    mean = cv2.blur(gray, (kernel_size, kernel_size))
    sqr_mean = cv2.blur(gray**2, (kernel_size, kernel_size))
    variance = sqr_mean - mean**2
    
    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    edge_mag = np.sqrt(sobelx**2 + sobely**2)
    
    laplacian = np.abs(cv2.Laplacian(gray, cv2.CV_32F))
    
    return {
        'variance': variance,
        'edge_mag': edge_mag,
        'laplacian': laplacian,
    }

def extract_enhanced_features(image, grid_size):
    h, w = image.shape[:2]
    gh, gw = grid_size
    
    patch_h = h // gh
    patch_w = w // gw
    
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    veg_indices = compute_vegetation_indices(image)
    texture = compute_texture_features(image)
    
    features = []
    for i in range(gh):
        for j in range(gw):
            y1, y2 = i * patch_h, (i + 1) * patch_h
            x1, x2 = j * patch_w, (j + 1) * patch_w
            
            patch_hsv = hsv[y1:y2, x1:x2]
            
            h_mean, s_mean, v_mean = patch_hsv.mean(axis=(0, 1))
            h_std, s_std, v_std = patch_hsv.std(axis=(0, 1))
            h_channel = patch_hsv[:, :, 0]
            green_ratio = ((h_channel >= 40) & (h_channel <= 85)).mean()
            dead_ratio = (h_channel < 40).mean()
            
            veg_feats = []
            for name, idx_map in veg_indices.items():
                patch_idx = idx_map[y1:y2, x1:x2]
                veg_feats.extend([patch_idx.mean(), patch_idx.std()])
            
            tex_feats = []
            for name, tex_map in texture.items():
                patch_tex = tex_map[y1:y2, x1:x2]
                tex_feats.append(patch_tex.mean())
            
            patch_features = [
                h_mean, h_std, s_mean, s_std, v_mean, v_std,
                green_ratio * 255, dead_ratio * 255,
            ] + veg_feats + tex_feats
            
            features.append(patch_features)
    
    return np.array(features)

def upsample_features(features, grid_size, factor=4):
    gh, gw = grid_size
    feat_dim = features.shape[1]
    
    feat_map = features.reshape(gh, gw, feat_dim)
    new_h, new_w = gh * factor, gw * factor
    upsampled = np.zeros((new_h, new_w, feat_dim))
    
    for c in range(feat_dim):
        upsampled[:, :, c] = cv2.resize(
            feat_map[:, :, c], 
            (new_w, new_h),
            interpolation=cv2.INTER_LINEAR
        )
    
    return upsampled.reshape(-1, feat_dim), (new_h, new_w)

def cluster_patches(features, n_clusters=8):
    scaler = StandardScaler()
    features_norm = scaler.fit_transform(features)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features_norm)
    return labels, kmeans

def post_process_segmentation(label_map, grid_size):
    labels_2d = label_map.reshape(grid_size).astype(np.float32)
    labels_smooth = median_filter(labels_2d, size=3)
    return labels_smooth.flatten().astype(np.int32)

def analyze_clusters_v4(image, labels, grid_size):
    """
    FIXED classification with TWO-STEP approach:
    Step 1: Is it vegetation? (ExG > 0.05 OR TGI > 0.03) AND (S > 40)
    Step 2: If vegetation, is it Green or Dead?
    """
    h, w = image.shape[:2]
    gh, gw = grid_size
    
    patch_h = h // gh
    patch_w = w // gw
    
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    veg_indices = compute_vegetation_indices(image)
    
    labels_2d = labels.reshape(gh, gw)
    
    cluster_info = {}
    for cluster_id in range(CFG.n_clusters):
        mask = labels_2d == cluster_id
        if mask.sum() == 0:
            continue
        
        h_vals, s_vals, v_vals = [], [], []
        exg_vals, tgi_vals, ngrdi_vals = [], [], []
        
        for i in range(gh):
            for j in range(gw):
                if labels_2d[i, j] == cluster_id:
                    y1, y2 = i * patch_h, (i + 1) * patch_h
                    x1, x2 = j * patch_w, (j + 1) * patch_w
                    
                    patch = hsv[y1:y2, x1:x2]
                    h_vals.append(patch[:, :, 0].mean())
                    s_vals.append(patch[:, :, 1].mean())
                    v_vals.append(patch[:, :, 2].mean())
                    
                    exg_vals.append(veg_indices['ExG'][y1:y2, x1:x2].mean())
                    tgi_vals.append(veg_indices['TGI'][y1:y2, x1:x2].mean())
                    ngrdi_vals.append(veg_indices['NGRDI'][y1:y2, x1:x2].mean())
        
        cluster_info[cluster_id] = {
            'H': np.mean(h_vals),
            'S': np.mean(s_vals),
            'V': np.mean(v_vals),
            'ExG': np.mean(exg_vals),
            'TGI': np.mean(tgi_vals),
            'NGRDI': np.mean(ngrdi_vals),
            'count': mask.sum(),
            'percent': mask.sum() / (gh * gw) * 100
        }
    
    # ADAPTIVE THRESHOLDS using percentiles
    # Collect ExG/TGI from all non-shadow clusters
    veg_exg = [v['ExG'] for v in cluster_info.values() if v['V'] >= 70]
    veg_tgi = [v['TGI'] for v in cluster_info.values() if v['V'] >= 70]
    
    if len(veg_exg) > 0:
        exg_green_thr = np.percentile(veg_exg, 60)
        exg_dead_thr = np.percentile(veg_exg, 35)
        tgi_green_thr = np.percentile(veg_tgi, 60)
        tgi_dead_thr = np.percentile(veg_tgi, 35)
    else:
        exg_green_thr, exg_dead_thr = 0.15, 0.10
        tgi_green_thr, tgi_dead_thr = 0.08, 0.05
    
    print("\n=== Enhanced Cluster Analysis v5 (Adaptive) ===")
    print(f"Thresholds: ExG[green>{exg_green_thr:.3f}, dead<{exg_dead_thr:.3f}]")
    print(f"            TGI[green>{tgi_green_thr:.3f}, dead<{tgi_dead_thr:.3f}]")
    print("-" * 70)
    
    classifications = {}
    for cid, vals in sorted(cluster_info.items(), key=lambda x: -x[1]['percent']):
        h_val, s_val, v_val = vals['H'], vals['S'], vals['V']
        exg, tgi, ngrdi = vals['ExG'], vals['TGI'], vals['NGRDI']
        
        # STEP 1: Shadow check
        if v_val < 70:
            label = "SHADOW"
        # STEP 2: Is it vegetation? (ExG > 0.05 OR TGI > 0.03) AND (S > 40)
        elif not (((exg > 0.05) or (tgi > 0.03)) and (s_val > 40)):
            label = "SOIL"
        else:
            # STEP 3: GREEN vs DEAD using adaptive thresholds
            # GREEN: High ExG AND high TGI (above 60th percentile)
            if (exg >= exg_green_thr) and (tgi >= tgi_green_thr):
                label = "GREEN"
            # DEAD: Low ExG OR low TGI, plus yellowish hue
            elif (20 <= h_val <= 55) and ((exg < exg_dead_thr) or (tgi < tgi_dead_thr) or (ngrdi < 0.03)):
                label = "DEAD"
            # CLOVER: Pink/purple
            elif (h_val < 20 or h_val > 150) and s_val > 60:
                label = "CLOVER"
            # Borderline: use ExG to decide
            elif exg >= 0.15:
                label = "GREEN"
            elif exg < 0.12 and h_val < 45:
                label = "DEAD"
            else:
                label = "GREEN"  # Default for ambiguous
        
        classifications[cid] = label
        cluster_info[cid]['label'] = label
        
        print(f"  Cluster {cid}: [{label:6s}]")
        print(f"    HSV: H={h_val:.1f}, S={s_val:.1f}, V={v_val:.1f}")
        print(f"    VegIdx: ExG={exg:.3f}, TGI={tgi:.3f}, NGRDI={ngrdi:.3f}")
        print(f"    Area: {vals['percent']:.1f}%")
    
    return cluster_info, classifications

def visualize_segmentation_v4(image, labels, grid_size, classifications, veg_indices, save_path):
    labels_2d = labels.reshape(grid_size)
    
    label_map = cv2.resize(
        labels_2d.astype(np.float32), 
        (image.shape[1], image.shape[0]),
        interpolation=cv2.INTER_NEAREST
    ).astype(np.int32)
    
    # Color map with SIMPLE strings (no emojis)
    color_map = {
        "GREEN": [0, 200, 0],
        "DEAD": [255, 200, 0],      # Yellow-orange for better visibility
        "CLOVER": [255, 105, 180],
        "SHADOW": [50, 50, 50],
        "SOIL": [139, 90, 43],      # Brown for soil
    }
    
    seg_colored = np.zeros_like(image)
    for cluster_id, label in classifications.items():
        mask = label_map == cluster_id
        color = color_map.get(label, [128, 128, 128])
        seg_colored[mask] = color
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    axes[0, 0].imshow(image)
    axes[0, 0].set_title("Original Image", fontsize=12)
    axes[0, 0].axis("off")
    
    axes[0, 1].imshow(seg_colored)
    axes[0, 1].set_title(f"Segmentation ({CFG.n_clusters} clusters)", fontsize=12)
    axes[0, 1].axis("off")
    
    overlay = cv2.addWeighted(image, 0.6, seg_colored, 0.4, 0)
    axes[0, 2].imshow(overlay)
    axes[0, 2].set_title("Overlay", fontsize=12)
    axes[0, 2].axis("off")
    
    exg_resized = cv2.resize(veg_indices['ExG'], (image.shape[1], image.shape[0]))
    axes[1, 0].imshow(exg_resized, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
    axes[1, 0].set_title("ExG (Excess Green)", fontsize=12)
    axes[1, 0].axis("off")
    
    tgi_resized = cv2.resize(veg_indices['TGI'], (image.shape[1], image.shape[0]))
    axes[1, 1].imshow(tgi_resized, cmap='RdYlGn', vmin=-0.3, vmax=0.3)
    axes[1, 1].set_title("TGI (Triangular Greenness)", fontsize=12)
    axes[1, 1].axis("off")
    
    # Create legend
    legend_img = np.ones((image.shape[0], image.shape[1], 3), dtype=np.uint8) * 255
    y_offset = 50
    for name, color in color_map.items():
        cv2.rectangle(legend_img, (20, y_offset-20), (60, y_offset+10), color, -1)
        cv2.putText(legend_img, name, (70, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 2)
        y_offset += 50
    axes[1, 2].imshow(legend_img)
    axes[1, 2].set_title("Legend", fontsize=12)
    axes[1, 2].axis("off")
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nSaved visualization to {save_path}")
    
    return label_map

def estimate_biomass_v4(cluster_info):
    print("\n" + "="*70)
    print("BIOMASS ESTIMATION v4:")
    print("="*70)
    
    green_pct = 0
    dead_pct = 0
    clover_pct = 0
    soil_pct = 0
    shadow_pct = 0
    
    for cid, info in cluster_info.items():
        pct = info['percent']
        label = info.get('label', '')
        
        if label == "GREEN":
            green_pct += pct
        elif label == "DEAD":
            dead_pct += pct
        elif label == "CLOVER":
            clover_pct += pct
        elif label == "SHADOW":
            shadow_pct += pct
        else:
            soil_pct += pct
    
    print(f"\n  RAW AREA:")
    print(f"    GREEN: {green_pct:.1f}%")
    print(f"    DEAD: {dead_pct:.1f}%")
    print(f"    CLOVER: {clover_pct:.1f}%")
    print(f"    SOIL: {soil_pct:.1f}%")
    print(f"    SHADOW: {shadow_pct:.1f}%")
    
    biomass_total = green_pct + dead_pct + clover_pct
    if biomass_total > 0:
        print(f"\n  NORMALIZED (biomass only):")
        print(f"    GREEN: {green_pct/biomass_total*100:.1f}%")
        print(f"    DEAD: {dead_pct/biomass_total*100:.1f}%")
        print(f"    CLOVER: {clover_pct/biomass_total*100:.1f}%")
    else:
        print("\n  No biomass detected (all soil/shadow)")
    
    return {
        'green': green_pct,
        'dead': dead_pct,
        'clover': clover_pct,
        'soil': soil_pct,
        'shadow': shadow_pct,
    }

def main():
    print("="*70)
    print("DINOv2 Pseudo-Segmentation v4 - FIXED")
    print("="*70)
    
    if not os.path.exists(CFG.image_path):
        train_dir = Path("/kaggle/input/csiro-biomass/train")
        if train_dir.exists():
            images = list(train_dir.glob("*.jpg"))
            if images:
                CFG.image_path = str(images[0])
        else:
            print("No images found")
            return
    
    print(f"\n1. Loading image: {CFG.image_path}")
    image = cv2.imread(CFG.image_path)
    if image is None:
        print("Could not load image")
        return
    
    h, w = image.shape[:2]
    left_half = image[:, :w//2, :]
    print(f"   Image shape: {left_half.shape}")
    
    print("\n2. Loading DINOv2...")
    model = load_dino_backbone()
    
    print("\n3. Extracting DINOv2 features...")
    dino_features, grid_size, img_rgb = extract_patch_features(model, left_half)
    print(f"   DINOv2: {dino_features.shape}, grid: {grid_size}")
    
    print("\n4. Computing vegetation indices...")
    veg_indices = compute_vegetation_indices(img_rgb)
    
    print("\n5. Extracting enhanced features...")
    enhanced_features = extract_enhanced_features(img_rgb, grid_size)
    print(f"   Enhanced: {enhanced_features.shape}")
    
    print("\n6. Combining features...")
    scaler = StandardScaler()
    dino_norm = scaler.fit_transform(dino_features)
    enhanced_norm = StandardScaler().fit_transform(enhanced_features)
    
    combined = np.concatenate([
        dino_norm * CFG.dino_weight,
        enhanced_norm * (1 - CFG.dino_weight)
    ], axis=1)
    
    print(f"\n7. Upsampling features ({CFG.upsample_factor}x)...")
    upsampled, new_grid = upsample_features(combined, grid_size, CFG.upsample_factor)
    
    print(f"\n8. Clustering (K={CFG.n_clusters})...")
    labels, kmeans = cluster_patches(upsampled, n_clusters=CFG.n_clusters)
    
    print("\n9. Post-processing...")
    labels = post_process_segmentation(labels, new_grid)
    
    print("\n10. Analyzing clusters (Two-Step)...")
    labels_small = cv2.resize(
        labels.reshape(new_grid).astype(np.float32), 
        (grid_size[1], grid_size[0]),
        interpolation=cv2.INTER_NEAREST
    ).astype(np.int32).flatten()
    
    cluster_info, classifications = analyze_clusters_v4(img_rgb, labels_small, grid_size)
    
    print("\n11. Visualizing...")
    save_path = "/kaggle/working/segmentation_v4.png"
    visualize_segmentation_v4(img_rgb, labels, new_grid, classifications, veg_indices, save_path)
    
    print("\n12. Estimating biomass...")
    biomass = estimate_biomass_v4(cluster_info)
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg

    img_result = mpimg.imread('/kaggle/working/segmentation_v4.png')
    plt.figure(figsize=(12, 8))
    plt.imshow(img_result)
    plt.axis('off')
    plt.title("Segmentación de Biomasa V4")
    plt.show()
    
    return biomass, cluster_info

if __name__ == "__main__":
    main()